# Wikidata Candidate Sense Merge Experiment

This notebook tests whether an OpenAI API LLM can merge Wikidata candidate senses that are semantically close or duplicated. It uses the same candidate-loading utility used by `LiteSemRAG` semantic-description assignment, then asks the LLM to produce a smaller cleaned candidate set.

## Parameters

Change `WORD` and `MAX_CANDIDATE_COUNT`, then run the notebook top to bottom. The OpenAI key is read from `OPENAI_API_KEY` if present, otherwise from the root `API_KEY` file.

In [22]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import build_wikidata_candidate_bank, load_wikidata_definition_candidates

MAX_CANDIDATE_COUNT = 8
USE_DETAILED_DESCRIPTION = False
USE_WIKIDATA_SPAN_RULES = False

OPENAI_MODEL = "gpt-5.4-mini"
API_KEY_PATH = REPO_ROOT / "API_KEY"


def load_local_api_config(path: Path = API_KEY_PATH) -> dict:
    if not path.exists():
        return {}
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return {}
    if text.startswith("{"):
        return json.loads(text)
    if "=" not in text:
        return {"OPENAI_API_KEY": clean_api_value(text)}
    config = {}
    for part in text.replace(";", "\n").splitlines():
        part = part.strip()
        if not part or part.startswith("#") or "=" not in part:
            continue
        key, value = part.split("=", 1)
        config[key.strip().strip("'\"")] = clean_api_value(value)
    return config


def clean_api_value(value: str) -> str:
    value = str(value).strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    value = value.replace("\\n", "").replace("\\r", "").strip()
    value = value.removesuffix(",").strip()
    value = value.strip("'\"")
    return value


def first_config_value(config: dict, *keys: str):
    for key in keys:
        value = config.get(key)
        if value:
            return value
    return None


LOCAL_API_CONFIG = load_local_api_config()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_API_KEY",
    "openai_api_key",
    "api_key",
    "chatgpt_api",
)
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or first_config_value(
    LOCAL_API_CONFIG,
    "OPENAI_BASE_URL",
    "openai_base_url",
    "base_url",
)

print(f"repo: {REPO_ROOT}")
print(f"local API key file exists: {API_KEY_PATH.exists()}")
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)}")
print(f"OpenAI base URL: {OPENAI_BASE_URL or '(default)'}")

repo: /home/xiaoyue/LiteSemRAG
local API key file exists: True
OpenAI key loaded: True
OpenAI base URL: (default)


## 1. Fetch Wikidata Candidate Senses

`MAX_CANDIDATE_COUNT` is passed both as the Wikidata search limit and the post-filter target count, so this cell requests up to that many usable definitions. Set `USE_WIKIDATA_SPAN_RULES = False` to use raw Wikidata candidates without the span-aware label/alias rules.

In [23]:
def fetch_candidate_senses(
    word: str,
    max_candidates: int,
    use_detailed_description: bool = False,
    use_span_rules: bool = True,
):
    candidates_df, definition_column = load_wikidata_definition_candidates(
        word,
        use_detailed_description=use_detailed_description,
        limit=max_candidates,
        target_candidate_count=max_candidates,
        use_span_rules=use_span_rules,
    )
    candidate_bank = build_wikidata_candidate_bank(
        candidates_df,
        definition_column=definition_column,
    )
    rows = []
    for idx, candidate in enumerate(candidate_bank, start=1):
        rows.append(
            {
                "candidate_id": idx,
                "wikidata_id": candidate["entity_id"],
                "label": candidate["label"],
                "description": candidate["description"],
                "definition": candidate["definition"],
                "hypothesis": candidate["hypothesis"],
            }
        )
    return rows, candidates_df

WORD = "space"
candidate_senses, raw_candidates_df = fetch_candidate_senses(
    WORD,
    MAX_CANDIDATE_COUNT,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    use_span_rules=False,
)

candidate_df = pd.DataFrame(candidate_senses)
candidate_df

,candidate_id,wikidata_id,label,description,definition,hypothesis
0,1,Q1,universe,"totality consisting of space, time, matter and...","totality consisting of space, time, matter and...","It refers to totality consisting of space, tim..."
1,2,Q380933,space,"blank area that separates words, sentences, sy...","blank area that separates words, sentences, sy...","It refers to blank area that separates words, ..."
2,3,Q4169,outer space,void between celestial bodies,void between celestial bodies,It refers to void between celestial bodies.
3,4,Q61977583,Space,2014 video game,2014 video game,It refers to 2014 video game.
4,5,Q63968426,S P A C E,"hackspace in Nuremberg, Germany","hackspace in Nuremberg, Germany","It refers to hackspace in Nuremberg, Germany."
5,6,Q107,space,three-dimensional extent in which objects exis...,three-dimensional extent in which objects exis...,It refers to three-dimensional extent in which...
6,7,Q193701,SpaceX,American private aerospace company,American private aerospace company,It refers to American private aerospace company.
7,8,Q16555,Houston,"seat of Harris County, and largest city in Sta...","seat of Harris County, and largest city in Sta...","It refers to seat of Harris County, and larges..."


## 2. Prompt The LLM To Merge Similar Senses

The prompt now biases the model toward coarse retrieval-oriented senses: merge institution/role variants that a cross-encoder would struggle to separate, and split only when the meanings would clearly retrieve different facts.

In [24]:
SYSTEM_PROMPT = """You are a conservative lexical-sense merger for a semantic retrieval system.
Your job is to reduce noisy Wikidata candidate senses to a small set of coarse retrieval meanings.
Prefer merging over splitting when candidates describe the same broad concept, role, entity type, or function.
Do not preserve fine-grained domain, institution, jurisdiction, title, or wording differences unless they change what evidence should be retrieved.
When uncertain, merge the candidates and write a broader description.
Return only valid JSON."""


MERGE_EXAMPLES = [
    {
        "word": "president",
        "candidate_senses": [
            {"candidate_id": 1, "label": "president", "hypothesis": "It refers to leader of a country or part of a country."},
            {"candidate_id": 2, "label": "president", "hypothesis": "It refers to leader of an organization."},
            {"candidate_id": 3, "label": "speaker", "hypothesis": "It refers to presiding officer of a legislative body."},
            {"candidate_id": 4, "label": "chancellor", "hypothesis": "It refers to leader of a university or college."},
        ],
        "expected_merge": {
            "canonical_label": "leader or presiding officer",
            "merged_description": "A person who leads or presides over a country, organization, legislative body, university, or similar institution.",
            "source_candidate_ids": [1, 2, 3, 4],
        },
        "rationale": "These are institutional leadership or presiding roles. The differences are title/domain variants, not separate retrieval meanings for a generic word-sense index.",
    },
    {
        "word": "bank",
        "candidate_senses": [
            {"candidate_id": 1, "label": "bank", "hypothesis": "It refers to a financial institution."},
            {"candidate_id": 2, "label": "river bank", "hypothesis": "It refers to land alongside a river."},
        ],
        "expected_split": [1, 2],
        "rationale": "These meanings retrieve different kinds of evidence and should stay separate.",
    },
    {
        "word": "apple",
        "candidate_senses": [
            {"candidate_id": 1, "label": "apple", "hypothesis": "It refers to an edible fruit."},
            {"candidate_id": 2, "label": "Apple Inc.", "hypothesis": "It refers to a technology company."},
        ],
        "expected_split": [1, 2],
        "rationale": "A fruit and a company are different entity types and should stay separate.",
    },
]


def compact_candidates_for_prompt(candidates: list[dict]) -> list[dict]:
    return [
        {
            "candidate_id": candidate["candidate_id"],
            "label": candidate["label"],
            "hypothesis": candidate["hypothesis"],
        }
        for candidate in candidates
    ]


def build_merge_prompt(word: str, candidates: list[dict]) -> str:
    payload = {
        "word": word,
        "candidate_senses": compact_candidates_for_prompt(candidates),
    }
    return f"""Merge candidate senses for the target word into coarse retrieval-oriented meanings.

Goal:
Create the smallest useful sense inventory for retrieval. These merged descriptions will later be used by a cross-encoder, so avoid distinctions that are too subtle for short context snippets.

Default bias:
- Merge by broad semantic function, not by Wikidata entity granularity.
- Merge title/domain variants when they are instances of the same role or concept.
- Merge specific subtypes into their broader parent sense unless the subtype changes the entity type or expected evidence.
- If two candidates could both match the same ordinary sentence about the target word, merge them.
- When uncertain, merge.

Split only when:
1. The meanings are genuinely different entity types or concepts, such as fruit vs company or financial bank vs river bank.
2. Keeping them together would make clearly wrong documents look relevant.
3. The distinction is likely obvious from short local context, not just from specialist wording.

Discard only when:
1. The candidate is not a plausible sense of the target word.
2. The candidate is too vague to add value and cannot be merged into a broader valid sense.

Examples:
{json.dumps(MERGE_EXAMPLES, ensure_ascii=False, indent=2)}

Output only valid JSON with keys: word, merged_senses, discarded_candidates, notes.

Each merged_senses item must contain:
- sense_id: a short stable id such as s1, s2, s3
- canonical_label: short label for the merged sense
- merged_description: one sentence describing the broad merged meaning
- source_candidate_ids: list of integer candidate_id values that were merged
- merge_rationale: one short sentence explaining why these candidates belong together or why the sense stayed separate

Each discarded_candidates item must contain:
- candidate_id
- reason

Candidate data:
{json.dumps(payload, ensure_ascii=False, indent=2)}"""


merge_prompt = build_merge_prompt(WORD, candidate_senses)
print(merge_prompt)

Merge candidate senses for the target word into coarse retrieval-oriented meanings.

Goal:
Create the smallest useful sense inventory for retrieval. These merged descriptions will later be used by a cross-encoder, so avoid distinctions that are too subtle for short context snippets.

Default bias:
- Merge by broad semantic function, not by Wikidata entity granularity.
- Merge title/domain variants when they are instances of the same role or concept.
- Merge specific subtypes into their broader parent sense unless the subtype changes the entity type or expected evidence.
- If two candidates could both match the same ordinary sentence about the target word, merge them.
- When uncertain, merge.

Split only when:
1. The meanings are genuinely different entity types or concepts, such as fruit vs company or financial bank vs river bank.
2. Keeping them together would make clearly wrong documents look relevant.
3. The distinction is likely obvious from short local context, not just from speci

In [25]:
from openai import OpenAI


def make_openai_client():
    client_kwargs = {}
    if OPENAI_API_KEY:
        client_kwargs["api_key"] = OPENAI_API_KEY
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    return OpenAI(**client_kwargs)


def merge_candidate_senses_with_llm(
    word: str,
    candidates: list[dict],
    model: str = OPENAI_MODEL,
):
    if not OPENAI_API_KEY:
        raise RuntimeError("OpenAI API key was not found in OPENAI_API_KEY or the root API_KEY file.")

    client = make_openai_client()
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_merge_prompt(word, candidates)},
        ],
    )
    content = response.choices[0].message.content
    return json.loads(content), response


llm_result, raw_response = merge_candidate_senses_with_llm(WORD, candidate_senses)
usage = raw_response.usage
token_usage = {
    "prompt_tokens": getattr(usage, "prompt_tokens", None),
    "completion_tokens": getattr(usage, "completion_tokens", None),
    "total_tokens": getattr(usage, "total_tokens", None),
}
print(token_usage)
llm_result

{'prompt_tokens': 1314, 'completion_tokens': 284, 'total_tokens': 1598}


{'word': 'space',
 'merged_senses': [{'sense_id': 's1',
   'canonical_label': 'physical space or universe',
   'merged_description': 'The physical expanse in which objects, events, matter, and energy exist, including outer space and the universe as a whole.',
   'source_candidate_ids': [1, 3, 6],
   'merge_rationale': 'These are closely related broad physical-cosmological meanings that ordinary context can overlap.'},
  {'sense_id': 's2',
   'canonical_label': 'typographical space',
   'merged_description': 'A blank character or area used to separate words, sentences, syllables, or other written symbols.',
   'source_candidate_ids': [2],
   'merge_rationale': 'This is a distinct writing-system meaning with different retrieval evidence.'},
  {'sense_id': 's3',
   'canonical_label': 'named entity or organization',
   'merged_description': 'A proper name referring to a specific entity such as a game, a hackspace, or a company.',
   'source_candidate_ids': [4, 5, 7],
   'merge_rationale': 

## 3. Inspect The Merged Candidate Set

In [26]:
merged_df = pd.DataFrame(llm_result.get("merged_senses", []))
merged_df

,sense_id,canonical_label,merged_description,source_candidate_ids,merge_rationale
0,s1,physical space or universe,"The physical expanse in which objects, events,...","[1, 3, 6]",These are closely related broad physical-cosmo...
1,s2,typographical space,A blank character or area used to separate wor...,[2],This is a distinct writing-system meaning with...
2,s3,named entity or organization,A proper name referring to a specific entity s...,"[4, 5, 7]",These are all named entities rather than the c...


In [27]:
discarded_df = pd.DataFrame(llm_result.get("discarded_candidates", []))
discarded_df

,candidate_id,reason
0,8,Not a plausible sense of the target word 'space'.


## 4. Save Experiment Output

In [28]:
# OUTPUT_DIR = REPO_ROOT / "cache" / "wikidata_llm_candidate_merge"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# output_path = OUTPUT_DIR / f"{WORD.replace(' ', '_')}_max{MAX_CANDIDATE_COUNT}_merged.json"
#
# record = {
#     "word": WORD,
#     "max_candidate_count": MAX_CANDIDATE_COUNT,
#     "use_detailed_description": USE_DETAILED_DESCRIPTION,
#     "openai_model": OPENAI_MODEL,
#     "token_usage": token_usage,
#     "raw_candidates": candidate_senses,
#     "llm_result": llm_result,
# }
# output_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
# print(output_path)